# Multi Vector IoT Botnet Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import networkx as nx
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy, zscore
import hashlib
from collections import defaultdict, deque

class IoTBotnetDetectionSystem:
    def __init__(self, device_threshold=1000, anomaly_window=300):
        self.device_graph = nx.DiGraph()
        self.traffic_classifier = RandomForestClassifier(n_estimators=150, random_state=42)
        self.behavioral_detector = IsolationForest(contamination=0.08, random_state=42)
        self.clustering_model = DBSCAN(eps=0.3, min_samples=5)
        self.scaler = StandardScaler()
        self.device_profiles = defaultdict(lambda: {'baseline': [], 'recent_activity': deque(maxlen=100)})
        self.botnet_signatures = defaultdict(list)
        self.anomaly_window = anomaly_window

    def extract_network_flow_features(self, network_flows):
        features = []
        for flow in network_flows:
            packet_sizes = np.array(flow.get('packet_sizes', [64]))
            inter_arrival_times = np.array(flow.get('inter_arrival_times', [0.1]))

            flow_duration = flow.get('duration', 1.0)
            total_bytes = sum(packet_sizes)
            packet_count = len(packet_sizes)

            size_entropy = entropy(np.histogram(packet_sizes, bins=10)[0] + 1e-10)
            timing_variance = np.var(inter_arrival_times)
            bandwidth_utilization = total_bytes / flow_duration

            protocol_diversity = len(set(flow.get('protocols', ['TCP'])))
            port_entropy = entropy(np.histogram(flow.get('dst_ports', [80]), bins=20)[0] + 1e-10)

            flow_asymmetry = abs(flow.get('upstream_bytes', 0) - flow.get('downstream_bytes', 0)) / max(total_bytes, 1)
            periodic_score = self.detect_periodic_behavior(inter_arrival_times)

            features.append([flow_duration, total_bytes, packet_count, size_entropy, timing_variance,
                           bandwidth_utilization, protocol_diversity, port_entropy, flow_asymmetry, periodic_score])

        return np.array(features)

    def detect_periodic_behavior(self, timestamps):
        if len(timestamps) < 5:
            return 0

        intervals = np.diff(timestamps)
        if len(intervals) == 0:
            return 0

        interval_hist = np.histogram(intervals, bins=min(10, len(intervals)))[0]
        periodicity_score = max(interval_hist) / sum(interval_hist) if sum(interval_hist) > 0 else 0

        autocorr = np.correlate(intervals, intervals, mode='full')
        autocorr_normalized = autocorr / max(autocorr.max(), 1e-10)
        periodic_peaks = len([x for x in autocorr_normalized if x > 0.7])

        return min(periodicity_score + periodic_peaks * 0.1, 1.0)

    def analyze_device_behavior(self, device_logs):
        behavioral_features = []

        for device_id, logs in device_logs.items():
            connection_count = len(logs.get('connections', []))
            unique_destinations = len(set(logs.get('destinations', [])))
            command_frequency = len(logs.get('commands_executed', [])) / max(logs.get('uptime', 1), 1)

            cpu_usage_pattern = np.array(logs.get('cpu_usage', [10]))
            memory_usage_pattern = np.array(logs.get('memory_usage', [20]))
            network_usage_pattern = np.array(logs.get('network_usage', [100]))

            cpu_anomaly = len(cpu_usage_pattern[zscore(cpu_usage_pattern) > 2]) / max(len(cpu_usage_pattern), 1)
            memory_spike_count = np.sum(np.diff(memory_usage_pattern) > 20)
            network_burst_ratio = np.sum(network_usage_pattern > np.percentile(network_usage_pattern, 90)) / len(network_usage_pattern)

            failed_auth_attempts = logs.get('failed_authentications', 0)
            privilege_escalation_attempts = logs.get('privilege_escalations', 0)

            behavioral_features.append([connection_count, unique_destinations, command_frequency,
                                      cpu_anomaly, memory_spike_count, network_burst_ratio,
                                      failed_auth_attempts, privilege_escalation_attempts])

            self.device_profiles[device_id]['recent_activity'].append({
                'timestamp': logs.get('timestamp', 0),
                'anomaly_score': cpu_anomaly + network_burst_ratio
            })

        return np.array(behavioral_features)

    def detect_ddos_coordination(self, network_traffic):
        coordination_indicators = []

        target_analysis = defaultdict(list)
        for packet in network_traffic:
            target = packet.get('dst_ip', '0.0.0.0')
            timestamp = packet.get('timestamp', 0)
            size = packet.get('size', 0)
            target_analysis[target].append({'time': timestamp, 'size': size, 'src': packet.get('src_ip', '')})

        for target, packets in target_analysis.items():
            if len(packets) > 100:
                timestamps = [p['time'] for p in packets]
                sources = set([p['src'] for p in packets])

                time_clustering = self.analyze_temporal_clustering(timestamps)
                source_diversity = len(sources) / len(packets)
                volume_spike = sum([p['size'] for p in packets]) / len(packets)

                coordination_score = time_clustering * 0.4 + source_diversity * 0.3 + min(volume_spike/1000, 0.3)
                coordination_indicators.append({'target': target, 'score': coordination_score, 'sources': len(sources)})

        return sorted(coordination_indicators, key=lambda x: x['score'], reverse=True)

    def analyze_temporal_clustering(self, timestamps):
        if len(timestamps) < 10:
            return 0

        timestamps = np.sort(timestamps)
        intervals = np.diff(timestamps)

        clustering_score = 0
        window_size = len(intervals) // 5

        for i in range(0, len(intervals) - window_size, window_size):
            window_intervals = intervals[i:i+window_size]
            if len(window_intervals) > 0:
                coefficient_variation = np.std(window_intervals) / (np.mean(window_intervals) + 1e-10)
                clustering_score += 1 / (1 + coefficient_variation)

        return clustering_score / max((len(intervals) // window_size), 1)

    def identify_c2_communications(self, communication_logs):
        c2_candidates = []

        communication_graph = nx.DiGraph()
        for log in communication_logs:
            src, dst = log.get('src_ip', ''), log.get('dst_ip', '')
            communication_graph.add_edge(src, dst, **log)

        centrality_scores = nx.betweenness_centrality(communication_graph)

        for node, centrality in centrality_scores.items():
            if centrality > 0.1:
                in_degree = communication_graph.in_degree(node)
                out_degree = communication_graph.out_degree(node)

                encrypted_ratio = 0
                total_connections = 0
                for _, _, data in communication_graph.edges(node, data=True):
                    total_connections += 1
                    if data.get('encrypted', False) or data.get('port', 80) in [443, 993, 995]:
                        encrypted_ratio += 1

                encrypted_ratio = encrypted_ratio / max(total_connections, 1)

                c2_probability = centrality * 0.4 + (in_degree / max(out_degree, 1)) * 0.3 + encrypted_ratio * 0.3
                c2_candidates.append({'ip': node, 'probability': c2_probability, 'centrality': centrality})

        return sorted(c2_candidates, key=lambda x: x['probability'], reverse=True)

    def comprehensive_botnet_analysis(self, network_flows, device_logs, communication_logs):
        flow_features = self.extract_network_flow_features(network_flows)
        behavioral_features = self.analyze_device_behavior(device_logs)

        ddos_coordination = self.detect_ddos_coordination(network_flows)
        c2_servers = self.identify_c2_communications(communication_logs)

        if len(flow_features) > 0:
            flow_anomalies = self.behavioral_detector.fit_predict(self.scaler.fit_transform(flow_features))
            flow_anomaly_ratio = np.mean(flow_anomalies == -1)
        else:
            flow_anomaly_ratio = 0

        if len(behavioral_features) > 0:
            device_clusters = self.clustering_model.fit_predict(self.scaler.fit_transform(behavioral_features))
            suspicious_clusters = len(set(device_clusters)) - 1 if -1 in device_clusters else len(set(device_clusters))
        else:
            suspicious_clusters = 0

        botnet_confidence = (
            flow_anomaly_ratio * 0.25 +
            min(len(ddos_coordination), 5) / 5 * 0.25 +
            min(len(c2_servers), 3) / 3 * 0.3 +
            min(suspicious_clusters, 10) / 10 * 0.2
        )

        threat_level = 'CRITICAL' if botnet_confidence > 0.8 else 'HIGH' if botnet_confidence > 0.6 else 'MEDIUM' if botnet_confidence > 0.4 else 'LOW'

        return {
            'botnet_detected': botnet_confidence > 0.5,
            'confidence': botnet_confidence,
            'threat_level': threat_level,
            'ddos_targets': len(ddos_coordination),
            'c2_servers': len(c2_servers),
            'infected_clusters': suspicious_clusters,
            'flow_anomaly_ratio': flow_anomaly_ratio,
            'top_c2_candidates': c2_servers[:3],
            'coordination_analysis': ddos_coordination[:5]
        }